# Mass mapping methods

We apply the three methods in SMPy, Kaiser-Squires, KS+ and aperture mass, to the same catalog, give the quantity
each one computes and its main settings, and compare the maps with the true convergence of a noiseless simulation,
with and without holes in the data.

## Settings

Paths are relative to `docs/source/tutorials` in the SMPy repository.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table
from scipy.ndimage import gaussian_filter

import smpy
from smpy.filters.processing import schirmer_filter, schneider_filter

DATA = Path("../../../examples/data")
CATALOG = DATA / "forecast_lum_annular.fits"
TRUTH = DATA / "forecast_lum_truth.fits"
SHEAR = dict(g1_col="g1_Rinv", g2_col="g2_Rinv", weight_col="weight")
PIXEL_SCALE = 0.4  # arcminutes
METHODS = {"kaiser_squires": "Kaiser-Squires", "ks_plus": "KS+", "aperture_mass": "Aperture mass"}

## Kaiser-Squires

In the weak-lensing limit, the shear $\gamma = \gamma_1 + i\gamma_2$ and the convergence $\kappa$ are related in
Fourier space by (Kaiser & Squires 1993)

$$
\hat\gamma(\mathbf{k}) = D(\mathbf{k})\,\hat\kappa(\mathbf{k}), \qquad
D(\mathbf{k}) = \frac{k_1^2 - k_2^2 + 2 i k_1 k_2}{k^2}.
$$

Since $|D| = 1$, the inversion is $\hat\kappa_E + i\hat\kappa_B = D^*\hat\gamma$. The E mode is the convergence, and
the B mode vanishes for pure lensing. The inversion assumes the shear is known everywhere on a periodic field, so
empty pixels and the field border leak into the map. Its one setting, `smoothing`, is the width in pixels of a
Gaussian applied to the result.

We recompute the unsmoothed map from the binned shear with NumPy. On a grid with an even side, SMPy also drops the
modes at the highest frequency of that side that vary along the other axis, where the sign of the $2 k_1 k_2$ term
is undefined.

In [ ]:
ks = smpy.map_mass(CATALOG, "kaiser_squires", pixel_scale=PIXEL_SCALE, smoothing=0, **SHEAR)

ky, kx = np.meshgrid(*(np.fft.fftfreq(n) for n in ks.grid.g1.shape), indexing="ij")
k_squared = np.where((kx == 0) & (ky == 0), 1, kx**2 + ky**2)
D = (kx**2 - ky**2 + 2j * kx * ky) / k_squared
D[((kx == -0.5) & (ky != 0)) | ((ky == -0.5) & (kx != 0))] = 0
kappa = np.fft.ifft2(np.conj(D) * np.fft.fft2(ks.grid.g1 + 1j * ks.grid.g2))
print("Largest difference from SMPy:", np.abs(kappa.real - ks.maps["E"]).max())

## KS+

KS+ (Pires et al. 2020) adds four corrections to Kaiser-Squires, each with its settings:

- **Missing data.** The shear in empty pixels is found by requiring the convergence to be sparse in the discrete
  cosine transform, over `inpainting_iterations` iterations, using the sparse inpainting of FASTLens
  (Pires et al. 2009) on which KS+ builds.
- **Field border.** The field is padded by `extension_size`, by default to twice its size, and the padding is
  treated as missing data.
- **Reduced shear.** Galaxies measure the reduced shear $g = \gamma / (1 - \kappa)$. KS+ reconstructs the map
  `reduced_shear_iterations` times, correcting the shear with the previous convergence.
- **Power spectrum.** Inside the gaps, the power of each starlet wavelet scale is matched to that outside
  (`use_wavelet_constraints`, `nscales`).

Without empty pixels, padding or reduced-shear correction, KS+ returns the Kaiser-Squires map.

## Aperture mass

The aperture mass at a pixel $\boldsymbol\theta_0$ is the tangential shear around it weighted by a filter $Q$
(Schneider 1996),

$$
M_\mathrm{ap}(\boldsymbol\theta_0) = \sum_j Q(|\boldsymbol\theta_j - \boldsymbol\theta_0|)\, \gamma_{t,j}, \qquad
\gamma_t = -\mathrm{Re}\left[\gamma\, e^{-2i\phi}\right],
$$

summed over the pixels within `truncation * filter_scale`, with $\phi$ the direction from $\boldsymbol\theta_0$ to
each pixel. SMPy provides the polynomial filter of Schneider et al. (1998) and the Schirmer filter as defined by
McCleary et al. (2020), which follows the tangential shear of an NFW halo. `filter_scale` is the aperture radius in
pixels and should match the size of the structures sought.

In [ ]:
radius = np.linspace(0, 1.2, 241)
fig, ax = plt.subplots(figsize=(6, 4))
for name, q in (("Schirmer", schirmer_filter(radius, 1)), ("Schneider, order 3", schneider_filter(radius, 1))):
    ax.plot(radius, q / q.max(), label=name)
ax.set_xlabel("Distance / filter_scale")
ax.set_ylabel("Q, normalized to its peak")
ax.legend()
plt.show()

We recompute the aperture mass at the highest pixel of the map from the binned shear.

In [ ]:
aperture = smpy.map_mass(CATALOG, "aperture_mass", pixel_scale=PIXEL_SCALE, **SHEAR)
scale = aperture.settings["methods"]["aperture_mass"]["filter_scale"]

y0, x0 = np.unravel_index(np.argmax(aperture.maps["E"]), aperture.maps["E"].shape)
y, x = np.indices(aperture.grid.g1.shape)
distance, phi = np.hypot(x - x0, y - y0), np.arctan2(y - y0, x - x0)
q = np.where((distance > 0) & (distance <= scale), schirmer_filter(distance / scale, scale), 0)
gamma_t = -(aperture.grid.g1 * np.cos(2 * phi) + aperture.grid.g2 * np.sin(2 * phi))
print(f"By hand: {np.sum(q * gamma_t):.6e}; SMPy: {aperture.maps['E'][y0, x0]:.6e}")

## The three methods on one catalog

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 5), gridspec_kw={"wspace": 0.45})
for ax, (method, name) in zip(axes, METHODS.items()):
    smpy.map_mass(CATALOG, method, pixel_scale=PIXEL_SCALE, **SHEAR).plot("E", ax=ax, cmap="magma", plot_title=name)
plt.show()

## Comparison with the true convergence

The truth catalog holds the noiseless reduced shear $g$ and magnification $\mu$ of the simulated cluster at each
galaxy, from which the convergence follows: $\mu = [(1 - \kappa)^2 (1 - |g|^2)]^{-1}$. Passing the convergence to
SMPy in place of $g_1$ bins it onto the pixels of the maps. We smooth it like the maps and remove the mean of every
map, which the shear does not determine.

In [ ]:
truth = Table.read(TRUTH, hdu=1)
truth["kappa"] = 1 - 1 / np.sqrt(truth["nfw_mu"] * (1 - truth["nfw_g1"]**2 - truth["nfw_g2"]**2))
truth["zero"] = 0.0
true_kappa = smpy.map_mass(truth, g1_col="kappa", g2_col="zero", pixel_scale=PIXEL_SCALE).grid.g1

kappa = gaussian_filter(true_kappa, 2.0)
kappa -= kappa.mean()
for method in ("kaiser_squires", "ks_plus"):
    convergence = smpy.map_mass(truth, method, g1_col="nfw_g1", g2_col="nfw_g2", pixel_scale=PIXEL_SCALE).maps["E"]
    convergence -= convergence.mean()
    print(f"{METHODS[method]}: correlation {np.corrcoef(convergence.ravel(), kappa.ravel())[0, 1]:.3f}, "
          f"RMS error {np.sqrt(np.mean((convergence - kappa)**2)):.4f} for a peak of {kappa.max():.3f}")

KS+ halves the error of Kaiser-Squires by correcting for reduced shear, which matters near the cluster center, and
for the field border.

## Holes in the data

Masks around bright stars leave holes in real catalogs. We give zero weight to the galaxies within 1.2 arcminutes
of eight random points and compare the unsmoothed maps with the truth inside the holes.

In [ ]:
rng = np.random.default_rng(3)
truth["weight"] = 1.0
for ra, dec in zip(rng.uniform(truth["ra"].min(), truth["ra"].max(), 8),
                   rng.uniform(truth["dec"].min(), truth["dec"].max(), 8)):
    offset = np.hypot((truth["ra"] - ra) * np.cos(np.deg2rad(dec)), truth["dec"] - dec)
    truth["weight"][offset < 0.02] = 0.0

fig, axes = plt.subplots(1, 2, figsize=(13, 4), layout="constrained")
for ax, method in zip(axes, ("kaiser_squires", "ks_plus")):
    result = smpy.map_mass(truth, method, g1_col="nfw_g1", g2_col="nfw_g2", weight_col="weight",
                           pixel_scale=PIXEL_SCALE, smoothing=0)
    observed, holes = result.grid.weights > 0, result.grid.weights == 0
    error = (result.maps["E"] - result.maps["E"][observed].mean()) - (true_kappa - true_kappa[observed].mean())
    image = ax.imshow(error, origin="lower", cmap="RdBu_r", vmin=-0.04, vmax=0.04)
    ax.contour(holes, levels=[0.5], colors="limegreen")
    ax.set_title(f"{METHODS[method]}: RMS error in the holes {np.sqrt(np.mean(error[holes]**2)):.4f}")
fig.colorbar(image, ax=axes, label="Map minus true convergence")
plt.show()